### CASE TECNICO PYSPARK

In [ ]:
##Import list
import pandas as pd
import matplotlib as plt
import os
import warnings
import logging
from pyspark.sql import SparkSession

# Suppress non-critical warnings
warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)

#set up global variables

CWD = os.getcwd()
CLIENTES_PATH = os.path.join(CWD, 'data/clients/data.json')
PEDIDOS_PATH = os.path.join(CWD, 'data/pedidos/data.json')

#start pyspark session

spark = (
    SparkSession.builder
    .appName("CaseTecnico")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.sql.autoBroadcastJoinThreshold", "10485760")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.default.parallelism", "8")
    .getOrCreate()
)

# Set log level AFTER session creation
spark.sparkContext.setLogLevel("ERROR")



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 07:32:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, LongType, StringType, StructType, StructField
from pyspark.storagelevel import StorageLevel

# explicit schemas avoid extra pass for inference
CLIENTES_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
])

PEDIDOS_SCHEMA = StructType([
    StructField("id", LongType(), True),
    StructField("client_id", LongType(), True),
    StructField("value", DecimalType(5, 2), True),
])


def load_data_from_json(path: str, schema: StructType, min_partitions: int | None = None) -> DataFrame:
    """Fast JSONL loader: schema-first, lazy, and optional repartition. Persists DataFrame in memory."""
    df = (
        spark.read
        .schema(schema)
        .option("multiLine", "false")
        .option("mode", "PERMISSIVE")
        .json(path)
    )

    if min_partitions is not None and df.rdd.getNumPartitions() < min_partitions:
        df = df.repartition(min_partitions)

    df = df.persist(StorageLevel.MEMORY_AND_DISK)
    return df



clientes_df = load_data_from_json(CLIENTES_PATH, CLIENTES_SCHEMA)
pedidos_df= load_data_from_json(PEDIDOS_PATH, PEDIDOS_SCHEMA, min_partitions=32)

print("clientes partitions:", clientes_df.rdd.getNumPartitions())
print("pedidos partitions:", pedidos_df.rdd.getNumPartitions())
print("schemas loaded successfully")

clientes partitions: 1
pedidos partitions: 32
schemas loaded successfully


## 1. Data Quality - Relatório de Falhas

In [ ]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Base columns (keep only what is needed)
pedidos_base_df = pedidos_df.select("id", "client_id", "value")
clientes_base_df = clientes_df.select("id", "name")

def regra_falha(df, motivo: str, ordem: int):
    return df.select(
        F.col("id"),
        F.lit(motivo).alias("motivo"),
        F.lit(ordem).alias("ordem_regra")
    )

# 1) Pedido sem valor
Sem_valor_ou_zero = regra_falha(
    pedidos_base_df.filter(F.col("value").isNull()),
    "pedido_sem_valor",
    1
)

# 3) ID de pedido duplicado
ids_duplicados_df = (
    pedidos_base_df
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .select("id")
)
ids_duplicados = regra_falha(ids_duplicados_df, "id_duplicado", 3)

# 4) Pedido com cliente inexistente (somente client_id válido)
cliente_inexistente = (
    pedidos_base_df.alias("p")
    .filter(F.col("p.client_id").isNotNull() & (F.col("p.client_id") > 0))
    .join(
        broadcast(clientes_base_df.select(F.col("id").alias("client_id_ref")).alias("c")),
        F.col("p.client_id") == F.col("c.client_id_ref"),
        "left_anti"
    )
    .select(F.col("p.id").alias("id"))
    .transform(lambda df: regra_falha(df, "cliente_inexistente", 4))
)

# 5) ID nulo
id_nullo = regra_falha(
    pedidos_base_df.filter(F.col("id").isNull()),
    "id_nulo",
    5
)

# 6) client_id nulo
client_id_nulo = regra_falha(
    pedidos_base_df.filter(F.col("client_id").isNull()),
    "client_id_nulo",
    6
)

# 7) ID inválido (<= 0)
id_invalido = regra_falha(
    pedidos_base_df.filter(F.col("id").isNotNull() & (F.col("id") <= 0)),
    "id_invalido_menor_igual_zero",
    7
)

# 8) client_id inválido (<= 0)
client_id_invalido = regra_falha(
    pedidos_base_df.filter(F.col("client_id").isNotNull() & (F.col("client_id") <= 0)),
    "client_id_invalido_menor_igual_zero",
    8
)

# 9) Valor zero
valor_zero = regra_falha(
    pedidos_base_df.filter(F.col("value") == 0),
    "valor_zero",
    9
)

# 10) Retorno sem pedido original correspondente
# Regra: para value < 0, deve existir (mesmo client_id, value positivo igual ao valor absoluto)
retornos_df = (
    pedidos_base_df
    .filter(F.col("value") < 0)
    .select(
        "id",
        "client_id",
        F.abs(F.col("value")).alias("valor_absoluto")
    )
)

pedidos_positivos_ref_df = (
    pedidos_base_df
    .filter(F.col("value") > 0)
    .select(
        F.col("client_id").alias("client_id_ref"),
        F.col("value").alias("valor_ref")
    )
    .distinct()
)

retornos_invalidos = (
    retornos_df.alias("r")
    .join(
        pedidos_positivos_ref_df.alias("p"),
        (F.col("r.client_id") == F.col("p.client_id_ref")) &
        (F.col("r.valor_absoluto") == F.col("p.valor_ref")),
        "left_anti"
    )
    .select(F.col("r.id").alias("id"))
    .transform(lambda df: regra_falha(df, "retorno_sem_pedido_original", 10))
)

# União de todas as regras
regras_falhas = [
    Sem_valor_ou_zero, ids_duplicados, cliente_inexistente, id_nullo, client_id_nulo,
    id_invalido, client_id_invalido, valor_zero, retornos_invalidos
]

falhas_df = (
    reduce(lambda acc, d: acc.unionByName(d), regras_falhas)
    .dropDuplicates(["id", "motivo"])
    .orderBy("ordem_regra", "id")
    .select("id", "motivo")
)

# Saídas pedidas
falhas_df.show(100, truncate=False)

erros_por_categoria_df = (
    falhas_df
    .groupBy("motivo")
    .agg(F.count("*").alias("qtd_erros"))
    .orderBy(F.col("qtd_erros").desc(), F.col("motivo").asc())
)

erros_por_categoria_df.show(truncate=False)
falhas_df.agg(F.count("*").alias("total_erros")).show()


+------+----------------+
|id    |motivo          |
+------+----------------+
|1534  |pedido_sem_valor|
|3502  |pedido_sem_valor|
|3679  |pedido_sem_valor|
|4322  |pedido_sem_valor|
|4724  |pedido_sem_valor|
|5774  |pedido_sem_valor|
|6322  |pedido_sem_valor|
|6622  |pedido_sem_valor|
|6884  |pedido_sem_valor|
|7116  |pedido_sem_valor|
|8597  |pedido_sem_valor|
|9322  |pedido_sem_valor|
|10786 |pedido_sem_valor|
|14240 |pedido_sem_valor|
|14804 |pedido_sem_valor|
|16194 |pedido_sem_valor|
|16383 |pedido_sem_valor|
|16929 |pedido_sem_valor|
|18559 |pedido_sem_valor|
|18623 |pedido_sem_valor|
|19401 |pedido_sem_valor|
|22785 |pedido_sem_valor|
|23468 |pedido_sem_valor|
|25638 |pedido_sem_valor|
|28249 |pedido_sem_valor|
|30342 |pedido_sem_valor|
|30377 |pedido_sem_valor|
|31027 |pedido_sem_valor|
|31045 |pedido_sem_valor|
|39954 |pedido_sem_valor|
|41080 |pedido_sem_valor|
|44260 |pedido_sem_valor|
|44409 |pedido_sem_valor|
|48490 |pedido_sem_valor|
|48920 |pedido_sem_valor|
|53800 |pedi

+-----------------------------------+---------+
|motivo                             |qtd_erros|
+-----------------------------------+---------+
|id_duplicado                       |54491    |
|pedido_sem_valor                   |49985    |
|retorno_sem_pedido_original        |24741    |
|client_id_invalido_menor_igual_zero|48       |
+-----------------------------------+---------+

+-----------+
|total_erros|
+-----------+
|     129265|
+-----------+



In [ ]:
from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# Base: only the columns we need; keep partitions aligned with the client join and cache for reuse
pedidos_base_df = (
    pedidos_df
    .select("id", "client_id", "value")
    .repartition(64, "client_id")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
total_pedidos = pedidos_base_df.count()

clientes_ids_df = (
    clientes_df
    .select(F.col("id").alias("client_id_ref"))
    .distinct()
)

# Aux 1: duplicated ids (any id that appears more than once)
ids_duplicados_df = (
    pedidos_base_df
    .groupBy("id")
    .agg(F.count("*").alias("dup_count"))
    .filter(F.col("dup_count") > 1)
    .select(F.col("id").alias("dup_id"))
)

# Aux 2: positive orders to validate returns (value < 0)
pedidos_positivos_ref_df = (
    pedidos_base_df
    .filter(F.col("value") > 0)
    .dropDuplicates(["client_id", "value"])
    .select(
        F.col("client_id").alias("ret_client_id_ref"),
        F.col("value").alias("ret_valor_ref"),
    )
)

# Enrich once with all info needed for filtering
enriched_pedidos_df = (
    pedidos_base_df.alias("p")
    # mark duplicated ids (broadcast small helper)
    .join(
        broadcast(ids_duplicados_df).alias("d"),
        F.col("p.id") == F.col("d.dup_id"),
        "left",
    )
    # attach client existence
    .join(
        broadcast(clientes_ids_df).alias("c"),
        F.col("p.client_id") == F.col("c.client_id_ref"),
        "left",
    )
    # attach matching positive order for potential returns
    .join(
        pedidos_positivos_ref_df.alias("rref"),
        (F.col("p.client_id") == F.col("rref.ret_client_id_ref"))
        & (F.abs(F.col("p.value")) == F.col("rref.ret_valor_ref")),
        "left",
    )
)

# Keep only rows that pass all quality rules
pedidos_validos_df = (
    enriched_pedidos_df
    .filter(
        # value present, non-zero, and within decimal(5,2) range
        F.col("p.value").isNotNull()
        & (F.col("p.value") != 0)
        & (F.col("p.value").cast(DecimalType(5, 2)).isNotNull())
        & (F.col("p.value") > 0)
        # valid id and client_id (not null, > 0)
        & F.col("p.id").isNotNull()
        & (F.col("p.id") >= 0)
        & F.col("p.client_id").isNotNull()
        & (F.col("p.client_id") >= 0)
        # not duplicated id
        & F.col("d.dup_id").isNull()
        # client exists for valid client_id
        & F.col("c.client_id_ref").isNotNull()
        # if value < 0, must have matching positive order; otherwise ok
    )
    .select(
        F.col("p.id").alias("id"),
        F.col("p.client_id").alias("client_id"),
        F.col("p.value").alias("value"),
    )
)
pedidos_validos_df = pedidos_validos_df.persist(StorageLevel.MEMORY_AND_DISK)
pedidos_validos = pedidos_validos_df.count()

print("Total pedidos (cached base):", total_pedidos)
print("Pedidos válidos (sem falhas nas regras):", pedidos_validos)

Total pedidos (cached base): 1100000
Pedidos válidos (sem falhas nas regras): 940493


In [ ]:
from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from pyspark.sql.window import Window


clientes_ids = [row['id'] for row in clientes_df.select("id").distinct().collect()]


pedidos_validos_df = (
    pedidos_df
    .select("id", "client_id", "value")
    .withColumn("is_valid_value", F.col("value").isNotNull() & (F.col("value") != 0) & (F.col("value").cast(DecimalType(5, 2)).isNotNull()) & (F.col("value") > 0))
    .withColumn("is_valid_id", F.col("id").isNotNull() & (F.col("id") >= 0))
    .withColumn("is_valid_client_id", F.col("client_id").isNotNull() & (F.col("client_id") >= 0))
    .withColumn("is_duplicated_id", F.count("id").over(Window.partitionBy("id")) > 1)
    .withColumn("is_valid_client", F.col("client_id").isin(clientes_ids))
    .filter(
        # value present, non-zero, and within decimal(5,2) range
        F.col("is_valid_value")
        # valid id and client_id (not null, > 0)
        & F.col("is_valid_id")
        & F.col("is_valid_client_id")
        # not duplicated id
        & ~F.col("is_duplicated_id")
        # valid client
        & F.col("is_valid_client")
    )
    .select("id", "client_id", "value")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

total_pedidos = pedidos_df.count()
pedidos_validos = pedidos_validos_df.count()

print("Total pedidos:", total_pedidos)
print("Pedidos válidos:", pedidos_validos)


Total pedidos: 1100000
Pedidos válidos: 940493


In [ ]:
from pyspark.sql.types import DecimalType, LongType
from pyspark.sql.functions import broadcast

cliente_totals_df = (
    pedidos_validos_df.alias("pedidos")
    .groupBy("client_id")
    .agg(
        F.count("*").cast(LongType()).alias("qtd_pedidos"),
        F.sum(F.col("value").cast(DecimalType(11, 2))).alias("valor_total"),
    ).join(
        broadcast(clientes_df.select(F.col("id").alias("client_id_ref"), 
                                     F.col("name").alias("client_name"))),
        F.col("pedidos.client_id") == F.col("client_id_ref"),
        "left"
    )
    .select(
        F.col("client_id").alias("id_cliente"),
        F.col("client_name").alias("nome_cliente"),
        F.col("qtd_pedidos"),
        F.col("valor_total")
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

cliente_totals_df.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|123456    |Inês Siqueira     |469734     |23698016.90|
|9047      |Zachary Reis      |61         |4002.08    |
|4494      |Vitor Marques     |68         |3937.06    |
|2695      |Wanda Silva       |61         |3736.53    |
|2756      |Inês Siqueira     |60         |3735.58    |
|8566      |Tereza Leal       |66         |3731.10    |
|6135      |Mariana Melo      |71         |3700.71    |
|5221      |Yasmin Carvalho   |65         |3679.35    |
|9266      |Tereza Leal       |67         |3678.17    |
|2379      |Gustavo Pontes    |65         |3657.84    |
|8317      |Sofia Castro      |66         |3656.86    |
|7543      |Vitória Andrade   |69         |3656.43    |
|849       |Breno Soares      |66         |3651.20    |
|6532      |João Batista      |68         |3650.20    |
|857       |Julio Viana       |62         |3645.

In [ ]:
media = cliente_totals_df.agg(F.mean("valor_total").alias("media_valor_total")).collect()[0]["media_valor_total"]

print(f"Valor médio total por cliente: {media:.2f}")

mediana = cliente_totals_df.approxQuantile("valor_total", [0.5], 0.01)[0]

print(f"Mediana do valor total por cliente: {mediana:.2f}")

percentil_10, percentil_90 = cliente_totals_df.approxQuantile("valor_total", [0.1, 0.9], 0.01)

print(f"10º percentil do valor total por cliente: {percentil_10:.2f}")

print(f"90º percentil do valor total por cliente: {percentil_90:.2f}")

Valor médio total por cliente: 4746.44


Mediana do valor total por cliente: 2358.82


10º percentil do valor total por cliente: 1866.70
90º percentil do valor total por cliente: 2862.07


In [ ]:
clientes_acima_media_df = (
    cliente_totals_df
    .filter(F.col("valor_total") > media)
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_acima_media_df.show(50, truncate=False)

+-------------+-----------+-----------+
|nome_cliente |qtd_pedidos|valor_total|
+-------------+-----------+-----------+
|Inês Siqueira|469734     |23698016.90|
+-------------+-----------+-----------+



In [ ]:
clientes_media_truncada_df = (
    cliente_totals_df
    .filter(
        (F.col("valor_total") >= percentil_10) &
        (F.col("valor_total") <= percentil_90)
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_media_truncada_df.show(50, truncate=False)

+------------------+-----------+-----------+
|nome_cliente      |qtd_pedidos|valor_total|
+------------------+-----------+-----------+
|Sebastião Toledo  |51         |2862.07    |
|Daniel Moreira    |59         |2861.99    |
|Zachary Reis      |60         |2861.80    |
|Diana Lima        |53         |2861.61    |
|Marcos Dias       |51         |2861.30    |
|Vitor Marques     |54         |2861.11    |
|Ana Silva         |58         |2861.10    |
|Diogo Tavares     |62         |2860.88    |
|Daniel Moreira    |54         |2860.64    |
|Oscar Vasconcelos |49         |2860.25    |
|Laura Gomes       |54         |2860.24    |
|Diogo Tavares     |51         |2860.21    |
|Vanessa Caldeira  |59         |2860.04    |
|Quintino Lira     |57         |2859.86    |
|Larissa Braga     |53         |2859.73    |
|Bernardo Pinto    |51         |2859.55    |
|Wagner Teixeira   |58         |2859.39    |
|Ximena Costa      |57         |2859.23    |
|Oscar Vasconcelos |57         |2859.19    |
|Breno Soa

##CASO ID 123456

os pedidos do id 123456, estao com o valor muito elevado e pode possivelmente ser um erro da base de dados, logo, vou explorar os valores relacionados a esse ID e executar os calculos e comparar os resultados.

In [ ]:
pedidos_ines = (
    pedidos_validos_df.alias("pedidos")
    .filter(F.col("client_id") == 123456)
    .orderBy(F.col("value").desc())
    .select("id", "value")
)

pedidos_ines.show(10,truncate=False)

quantidade_ines99 = pedidos_ines.filter(F.col("value") == 99.99).count()

print("Quantidade de pedidos de Inês com valor 99.99:", quantidade_ines99)

Ines_valores_repitidos_df = (
    pedidos_ines.groupBy("value")
    .agg(F.count("*").alias("qtd_repeticoes"))
    .filter(F.col("qtd_repeticoes") > 1)
    .orderBy(F.col("qtd_repeticoes").desc(), F.col("value").asc())
)

Ines_valores_repitidos_df.show(10, truncate=False)


+--------+-----+
|id      |value|
+--------+-----+
|56757517|99.99|
|83496750|99.99|
|85827365|99.99|
|1230531 |99.99|
|73735842|99.99|
|29038115|99.99|
|96638460|99.99|
|58270999|99.99|
|65425509|99.99|
|70102887|99.99|
+--------+-----+
only showing top 10 rows
Quantidade de pedidos de Inês com valor 99.99: 26


+-----+--------------+
|value|qtd_repeticoes|
+-----+--------------+
|99.72|75            |
|51.41|74            |
|59.25|74            |
|42.32|71            |
|51.94|71            |
|63.11|71            |
|66.57|71            |
|83.85|71            |
|84.41|71            |
|16.36|70            |
+-----+--------------+
only showing top 10 rows


In [ ]:
cliente_totals_df.filter(F.col("id_cliente") != 123456).show(truncate=False)   

+----------+---------------+-----------+-----------+
|id_cliente|nome_cliente   |qtd_pedidos|valor_total|
+----------+---------------+-----------+-----------+
|9047      |Zachary Reis   |61         |4002.08    |
|4494      |Vitor Marques  |68         |3937.06    |
|2695      |Wanda Silva    |61         |3736.53    |
|2756      |Inês Siqueira  |60         |3735.58    |
|8566      |Tereza Leal    |66         |3731.10    |
|6135      |Mariana Melo   |71         |3700.71    |
|5221      |Yasmin Carvalho|65         |3679.35    |
|9266      |Tereza Leal    |67         |3678.17    |
|2379      |Gustavo Pontes |65         |3657.84    |
|8317      |Sofia Castro   |66         |3656.86    |
|7543      |Vitória Andrade|69         |3656.43    |
|849       |Breno Soares   |66         |3651.20    |
|6532      |João Batista   |68         |3650.20    |
|857       |Julio Viana    |62         |3645.72    |
|9147      |Zachary Reis   |59         |3645.27    |
|2781      |Ivan Almeida   |73         |3641.0

In [ ]:
media_A = cliente_totals_df.filter(F.col("id_cliente") != 123456).agg(F.mean("valor_total").alias("media_valor_total")).collect()[0]["media_valor_total"]

print(f"Valor médio total por cliente: {media:.2f}")

mediana_A = cliente_totals_df.filter(F.col("id_cliente") != 123456).approxQuantile("valor_total", [0.5], 0.01)[0]

print(f"Mediana do valor total por cliente: {mediana:.2f}")

percentil_10_A, percentil_90_A = cliente_totals_df.filter(F.col("id_cliente") == 123456).approxQuantile("valor_total", [0.1, 0.9], 0.01)

print(f"10º percentil do valor total por cliente: {percentil_10:.2f}")

print(f"90º percentil do valor total por cliente: {percentil_90:.2f}")

Valor médio total por cliente: 2377.11


Mediana do valor total por cliente: 2362.39


10º percentil do valor total por cliente: 1886.57
90º percentil do valor total por cliente: 2865.47


In [ ]:
clientes_acima_media_df_A = (
    cliente_totals_df
    .filter(
        (F.col("id_cliente") != 123456) & (F.col("valor_total") > media_A)
        )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_acima_media_df_A.show(50, truncate=False)

+----------+------------------+-----------+-----------+
|id_cliente|nome_cliente      |qtd_pedidos|valor_total|
+----------+------------------+-----------+-----------+
|9047      |Zachary Reis      |61         |4002.08    |
|4494      |Vitor Marques     |68         |3937.06    |
|2695      |Wanda Silva       |61         |3736.53    |
|2756      |Inês Siqueira     |60         |3735.58    |
|8566      |Tereza Leal       |66         |3731.10    |
|6135      |Mariana Melo      |71         |3700.71    |
|5221      |Yasmin Carvalho   |65         |3679.35    |
|9266      |Tereza Leal       |67         |3678.17    |
|2379      |Gustavo Pontes    |65         |3657.84    |
|8317      |Sofia Castro      |66         |3656.86    |
|7543      |Vitória Andrade   |69         |3656.43    |
|849       |Breno Soares      |66         |3651.20    |
|6532      |João Batista      |68         |3650.20    |
|857       |Julio Viana       |62         |3645.72    |
|9147      |Zachary Reis      |59         |3645.

In [ ]:
clientes_media_truncada_df = (
    cliente_totals_df
    .filter(
        (F.col("id_cliente") != 123456) &
        (F.col("valor_total") >= percentil_10_A) &
        (F.col("valor_total") <= percentil_90_A)
    )
    .orderBy(F.col("valor_total").desc(), F.col("nome_cliente"))
)

clientes_media_truncada_df.show(50, truncate=False)

NameError: name 'percentil_10_A' is not defined